# 13a MindEye2's one-session models on the one-session test set

**Kaggle:** GPU (P100 or T4), Internet on. **Inputs:** none: MindEye2's checkpoints, betas and test images come from
the Hugging Face dataset `pscotti/mindeyev2` (NSD's terms apply).

Development analysis, not a prespecified test. For NSD subjects 1, 2, 5, 7: MindEye2's released one-session model,
its retrieval path only, on the 1,000 shared test images as their code runs it. Writes each person's brain-by-image
cosine matrix for 13b, both retrieval directions on 12b's sets, and MindEye2's own evaluation call, which must
reproduce their published 79.0 and 57.4 within 4 points.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '13a-mindeye2', 'gpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import sys
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"targets": ["subj01", "subj02", "subj05", "subj07"], "hf_repo": "pscotti/mindeyev2", "model": "final_{s}_pretrained_1sess_24bs", "hidden_dim": 4096, "n_blocks": 4, "clip_seq_dim": 256, "clip_emb_dim": 1664, "batch": 25, "clip_arch": "ViT-bigG-14", "clip_version": "laion2b_s39b_b160k", "open_clip": "2.24.0", "kornia": "0.7.1", "n_subsets": 30, "subset_size": 300, "subset_seed": 401, "their_seed": 42, "published": {"MindEye2 (1 session)": [79.0, 57.4], "MindTuner (1 session)": [83.1, 76.0], "MindEye2 (full data)": [98.8, 98.3]}, "reproduce_tolerance": 4.0, "cache": "/tmp/me2"}')

import numpy as np
from cross_subject_encoding.utils import config as C

import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"open_clip_torch=={FIXED['open_clip']}", f"kornia=={FIXED['kornia']}"], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("13a needs the GPU (ViT-bigG and MindEye2's backbone)")
print(torch.__version__, torch.cuda.get_device_name(0), flush=True)

In [ ]:
import torch.nn as nn
class BrainNetwork(nn.Module):
    def __init__(self, h=4096, in_dim=15724, out_dim=768, seq_len=2, n_blocks=4, drop=.15, clip_size=768, clip_scale=1):
        super().__init__()
        self.seq_len, self.h, self.clip_size, self.clip_scale = seq_len, h, clip_size, clip_scale
        self.mixer_blocks1 = nn.ModuleList([self.mixer_block1(h, drop) for _ in range(n_blocks)])
        self.mixer_blocks2 = nn.ModuleList([self.mixer_block2(seq_len, drop) for _ in range(n_blocks)])
        self.backbone_linear = nn.Linear(h * seq_len, out_dim, bias=True)
        self.clip_proj = self.projector(clip_size, clip_size, h=clip_size)
    def projector(self, in_dim, out_dim, h=2048):
        return nn.Sequential(nn.LayerNorm(in_dim), nn.GELU(), nn.Linear(in_dim, h), nn.LayerNorm(h), nn.GELU(),
                             nn.Linear(h, h), nn.LayerNorm(h), nn.GELU(), nn.Linear(h, out_dim))
    def mlp(self, in_dim, out_dim, drop):
        return nn.Sequential(nn.Linear(in_dim, out_dim), nn.GELU(), nn.Dropout(drop), nn.Linear(out_dim, out_dim))
    def mixer_block1(self, h, drop):
        return nn.Sequential(nn.LayerNorm(h), self.mlp(h, h, drop))
    def mixer_block2(self, seq_len, drop):
        return nn.Sequential(nn.LayerNorm(seq_len), self.mlp(seq_len, seq_len, drop))
    def forward(self, x):
        residual1, residual2 = x, x.permute(0, 2, 1)
        for block1, block2 in zip(self.mixer_blocks1, self.mixer_blocks2):
            x = block1(x) + residual1
            residual1 = x
            x = x.permute(0, 2, 1)
            x = block2(x) + residual2
            residual2 = x
            x = x.permute(0, 2, 1)
        x = x.reshape(x.size(0), -1)
        backbone = self.backbone_linear(x).reshape(len(x), -1, self.clip_size)
        return backbone, self.clip_proj(backbone)

class RidgeRegression(torch.nn.Module):
    def __init__(self, input_sizes, out_features):
        super().__init__()
        self.out_features = out_features
        self.linears = torch.nn.ModuleList([torch.nn.Linear(n, out_features) for n in input_sizes])
    def forward(self, x, subj_idx):
        return self.linears[subj_idx](x[:, 0]).unsqueeze(1)

BLURRY = ("blin1.", "bdropout.", "bnorm.", "bupsampler.", "b_maps_projector.")
def load_retrieval_path(state_dict, num_voxels):
    ridge = RidgeRegression([num_voxels], FIXED["hidden_dim"])
    backbone = BrainNetwork(h=FIXED["hidden_dim"], in_dim=FIXED["hidden_dim"], seq_len=1, n_blocks=FIXED["n_blocks"],
                            clip_size=FIXED["clip_emb_dim"], out_dim=FIXED["clip_emb_dim"] * FIXED["clip_seq_dim"])
    ridge.load_state_dict({k[len("ridge."):]: v for k, v in state_dict.items() if k.startswith("ridge.")}, strict=True)
    sd = {k[len("backbone."):]: v for k, v in state_dict.items() if k.startswith("backbone.")}
    missing, unexpected = backbone.load_state_dict(sd, strict=False)
    if missing or any(not k.startswith(BLURRY) for k in unexpected):
        raise ValueError(f"checkpoint does not match the retrieval path: missing {missing[:5]}, unexpected {[k for k in unexpected if not k.startswith(BLURRY)][:5]}")
    return ridge.eval().requires_grad_(False), backbone.eval().requires_grad_(False)

def brain_embeddings(ridge, backbone, voxel):
    # voxel (n_images, 3, n_voxels); recon_inference: each repetition through ridge and backbone, outputs averaged
    out = []
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.float16):
        for i in range(0, len(voxel), FIXED["batch"]):
            v = voxel[i:i + FIXED["batch"]].to(DEVICE)
            c = None
            for rep in range(3):
                _, c0 = backbone(ridge(v[:, [rep]], 0))
                c = c0 if c is None else c + c0
            out.append((c / 3).float().cpu())
    return torch.cat(out)

In [ ]:
from huggingface_hub import hf_hub_download
import kornia
import open_clip
DEVICE = "cuda"
CACHE = Path(FIXED["cache"])
CACHE.mkdir(parents=True, exist_ok=True)
get = lambda name: hf_hub_download(FIXED["hf_repo"], name, repo_type="dataset", cache_dir=str(CACHE))
def drop(path):  # the cache keeps a symlink to a blob; free the blob
    p = Path(path)
    p.resolve().unlink()
    p.unlink()
shared_mask = np.load(get("shared1000.npy"))
shared_ids = np.where(shared_mask)[0] if shared_mask.dtype == bool else np.sort(shared_mask.astype(int))
images = torch.load(get("evals/all_images.pt"), map_location="cpu")
print("all_images", tuple(images.shape), images.dtype, "range", float(images.min()), float(images.max()), "| shared1000", shared_mask.shape, int(shared_mask.sum()), flush=True)

# FrozenOpenCLIPImageEmbedder(arch ViT-bigG-14, laion2b_s39b_b160k, output_tokens=True, only_tokens=True), as MindEye2
clip_model, _, _ = open_clip.create_model_and_transforms(FIXED["clip_arch"], device=torch.device("cpu"), pretrained=FIXED["clip_version"], cache_dir=str(CACHE))
del clip_model.transformer
clip_model = clip_model.eval().requires_grad_(False).to(DEVICE)
clip_model.visual.output_tokens = True
MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=DEVICE)
STD = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=DEVICE)
def embed_images(x):
    out = []
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.float16):
        for i in range(0, len(x), FIXED["batch"]):
            b = x[i:i + FIXED["batch"]].to(DEVICE).float()
            b = kornia.geometry.resize(b, (224, 224), interpolation="bicubic", align_corners=True, antialias=True)
            b = kornia.enhance.normalize((b + 1.0) / 2.0, MEAN, STD)  # MindEye2's preprocess, kept as is
            _, tokens = clip_model.visual(b)
            out.append(tokens.float().cpu())
    return torch.cat(out)
img_emb = embed_images(images)
print("image tokens", tuple(img_emb.shape), "|", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)
del clip_model
torch.cuda.empty_cache()
img_norm = torch.nn.functional.normalize(img_emb.flatten(1), dim=-1)
del img_emb
import shutil
for p in CACHE.glob("models--laion--*"):  # the CLIP weights (10 GB) are no longer needed
    shutil.rmtree(p)

In [ ]:
import h5py
import io
import tarfile
SUBSETS = [np.sort(np.random.default_rng([FIXED["subset_seed"], j]).permutation(1000)[:FIXED["subset_size"]]) for j in range(FIXED["n_subsets"])]  # as 12b

def read_test_tar(path):
    # webdataset samples: <key>.behav.npy is (1, 17); column 0 the 73k image index, column 5 the betas row
    rows = []
    with tarfile.open(path) as tf:
        for m in tf.getmembers():
            if m.isfile() and m.name.split(".", 1)[1] == "behav.npy":
                b = np.load(io.BytesIO(tf.extractfile(m).read()))
                rows.append((int(b[0, 0]), int(b[0, 5])))
    return np.array(rows)

def retrieval(S):
    # S rows: brain responses, columns: images. image retrieval: each brain picks an image; brain retrieval: each image picks a brain
    img, brain = [], []
    for sub in SUBSETS:
        s = S[np.ix_(sub, sub)]
        eye = np.arange(len(sub))
        img.append((s.argmax(1) == eye).mean() * 100)
        brain.append((s.argmax(0) == eye).mean() * 100)
    return float(np.mean(img)), float(np.mean(brain))

def their_eval(brain_norm, img_norm):
    # final_evaluations.ipynb: 30 loops of np.random.choice(1000, 300); fwd = topk(batchwise_cosine_similarity(brain, clip)),
    # bwd = topk(batchwise_cosine_similarity(clip, brain)); utils.batchwise_cosine_similarity returns (Z @ B.T).T
    np.random.seed(FIXED["their_seed"])
    def batchwise(Z, B):
        return (Z @ B.T).T
    def topk1(sim):
        return float((sim.argmax(1) == torch.arange(len(sim))).float().mean())
    fwd, bwd = [], []
    for _ in range(30):
        r = np.random.choice(np.arange(1000), size=300, replace=False)
        fwd.append(topk1(batchwise(brain_norm[r], img_norm[r])) * 100)
        bwd.append(topk1(batchwise(img_norm[r], brain_norm[r])) * 100)
    return float(np.mean(fwd)), float(np.mean(bwd))

res = {"post_hoc": "development: MindEye2's released one-session models on the one-session test set", "people": {}}
test_ids = None
for s in FIXED["targets"]:
    t = time.monotonic()
    n = s[-1]
    tar = read_test_tar(get(f"wds/subj0{n}/new_test/0.tar"))
    b_path = get(f"betas_all_subj0{n}_fp32_renorm.hdf5")
    with h5py.File(b_path, "r") as f:
        betas = torch.from_numpy(f["betas"][:])
    drop(b_path)
    uniq = np.unique(tar[:, 0])
    if test_ids is None:
        test_ids = uniq
    if not (len(uniq) == 1000 and np.array_equal(uniq, test_ids) and np.array_equal(shared_ids, uniq)):
        raise ValueError(f"{s}: test images differ from shared1000 ({len(uniq)})")
    voxel = []
    for im in uniq:  # recon_inference's repeat rule
        locs = np.where(tar[:, 0] == im)[0]
        locs = np.repeat(locs, 3) if len(locs) == 1 else (np.tile(locs, 2)[:3] if len(locs) == 2 else locs)
        assert len(locs) == 3
        voxel.append(betas[tar[locs, 1]])
    voxel = torch.stack(voxel)
    del betas
    ck_path = get(f"train_logs/{FIXED['model'].format(s=s)}/last.pth")
    try:
        ck = torch.load(ck_path, map_location="cpu", mmap=True, weights_only=False)
    except RuntimeError:  # an old-format file cannot be memory-mapped
        ck = torch.load(ck_path, map_location="cpu", weights_only=False)
    ridge, backbone = load_retrieval_path(ck["model_state_dict"], voxel.shape[-1])
    del ck
    drop(ck_path)
    ridge, backbone = ridge.to(DEVICE), backbone.to(DEVICE)
    brain_norm = torch.nn.functional.normalize(brain_embeddings(ridge, backbone, voxel).flatten(1), dim=-1)
    del ridge, backbone
    torch.cuda.empty_cache()
    S = (brain_norm @ img_norm.T).numpy().astype(np.float32)
    np.save(C.METRICS_DIR / f"13a_{s}_cosine.npy", S)
    img, brain = retrieval(S)
    fwd, bwd = their_eval(brain_norm, img_norm)
    res["people"][s] = {"voxels": int(voxel.shape[-1]), "test_trials": int(len(tar)), "image_ret": img, "brain_ret": brain,
                        "their_fwd": fwd, "their_bwd": bwd}
    print(s, {k: (round(v, 1) if isinstance(v, float) else v) for k, v in res["people"][s].items()}, "|", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)
    del brain_norm, voxel
np.save(C.METRICS_DIR / "13a_test_ids.npy", test_ids)  # 0-based 73k indices, sorted

In [ ]:
mean = lambda k: float(np.mean([v[k] for v in res["people"].values()]))
res["means"] = {k: mean(k) for k in ("image_ret", "brain_ret", "their_fwd", "their_bwd")}
pub_img, pub_brain = FIXED["published"]["MindEye2 (1 session)"]
# which of our directions is their published 'image' number: their fwd uses (brain @ clip.T).T, i.e. each image picks a brain
res["their_fwd_is"] = "brain_ret" if abs(res["means"]["their_fwd"] - res["means"]["brain_ret"]) < abs(res["means"]["their_fwd"] - res["means"]["image_ret"]) else "image_ret"
tol = FIXED["reproduce_tolerance"]
res["reproduced"] = bool(abs(res["means"]["their_fwd"] - pub_img) <= tol and abs(res["means"]["their_bwd"] - pub_brain) <= tol)
(C.METRICS_DIR / "13a_summary.json").write_text(json.dumps(res, indent=2))
print("13A_SUMMARY_JSON " + json.dumps(res))
if not res["reproduced"]:
    print("NOT REPRODUCED: their evaluation gives", res["means"]["their_fwd"], res["means"]["their_bwd"], "against", pub_img, pub_brain)